# CV Masterclass 05: Vision Transformers (ViT)

Welcome to Notebook 05. For a decade, the Convolutional Neural Network (CNN) ruled Computer Vision unchallenged. 

Then, taking inspiration from the massive success of NLP Large Language Models, researchers asked: *"What if we just treat an image exactly like a sentence of words, and feed it into a standard Transformer?"*

---

## 🎓 Socratic Deep Check
Ponder this question before we begin:

> *"CNNs possess 'Inductive Bias'—a hardcoded mathematical assumption that pixels near each other are highly correlated. Vision Transformers have absolutely zero Inductive Bias. Why does this lack of mathematical assumption mean ViTs require 10x more training data than CNNs to achieve the same accuracy, but eventually allows them to drastically outperform CNNs when trained on 100x more data?"*

---

## Table of Contents
1. **The Death of Convolutions:** Breaking images into tokens.
2. **Patch Embeddings:** Linear projection of flattened grids.
3. **Position Embeddings:** Solving the Transformer's blindness to geometry.
4. **The `[CLS]` Token:** How a sequence predicts a single class.
5. **Inductive Bias:** Why scaling laws favor Transformers.


## 1. The Patching Paradigm

A Transformer expects a 1D sequence of tokens (words). An image is a 3D grid $(H, W, C)$. 

If we flatten every single pixel into a word, a $224\times224$ image becomes $50,176$ words. Transformer Self-Attention scales as $O(N^2)$. Calculating attention across 50,000 tokens requires $2.5$ Billion operations *per layer*. The GPU instantly crashes.

**The Solution:** We slice the image into larger $16\times16$ patches. 
A $224\times224$ image divided into $16\times16$ blocks gives exactly a $14\times14$ grid of patches. 

$14 \times 14 = 196$ patches.

We treat each of these $196$ patches as a "word" in a sentence. Now, $N=196$, which is incredibly fast for a Transformer.

In [ ]:
import torch
import torch.nn as nn

# -----------------------------------------------------
# IMPLEMENTATION: Flattening an Image into Patches
# -----------------------------------------------------

class PatchEmbedding(nn.Module):
    def __init__(self, in_channels=3, patch_size=16, embed_dim=768):
        super().__init__()
        self.patch_size = patch_size
        
        # The PyTorch Trick: A Conv2d layer with kernel_size == stride physically extracts
        # non-overlapping blocks of the image flawlessly and projects them instantly into the embedding dimension.
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # x shape: (Batch, 3, 224, 224)
        x = self.proj(x) 
        # x shape now: (Batch, 768, 14, 14)
        
        # Flatten the spatial dimensions (14x14 = 196)
        x = x.flatten(2) 
        # x shape now: (Batch, 768, 196)
        
        # Swap dimensions to match NLP Transformer format: (Batch, Sequence Length, Embed Dim)
        x = x.transpose(1, 2) 
        # x shape now: (Batch, 196, 768)
        
        return x

dummy_image = torch.randn(1, 3, 224, 224)
patcher = PatchEmbedding(patch_size=16, embed_dim=768)
tokens = patcher(dummy_image)

print(f"Original Image Shape: {dummy_image.shape}")
print(f"Sequence of Tokens Shape: {tokens.shape} -> (Batch, Tokens, Embedding_Dim)")
print("The image is now a sentence. Ready for Self-Attention.")

## 2. Position Embeddings

If you scramble the words in a sentence, a Convolutional Network will break, because it slides sequentially (it inherently understands $(x,y)$ space).

A Transformer has no idea what space is. If you scramble the 196 patches of a dog, the Transformer calculates the exact same attention vectors. It is "Permutation Invariant".

**The Fix:** Before passing the 196 tokens into the Transformer, we create 196 learnable parameters (a set of vectors). We mathematically `add` Vector 1 to Patch 1, Vector 2 to Patch 2, etc. The network slowly learns that Vector 1 implies "Top Left" and Vector 196 implies "Bottom Right".

We forcibly inject the coordinates back into the data.

## 3. The `[CLS]` Token

After the 196 tokens pass through the Attention layers, they come out as 196 updated tokens. 
How do we classify the image? Do we average all 196 tokens together? 

In ViT, we borrow a trick from BERT (NLP). At the very beginning of the sequence, before inserting it into the Transformer, we explicitly append a dummy, blank 197th token called the **Classification `[CLS]` Token**. 

The sequence length is now 197. As it passes through the multi-head attention layers, the `[CLS]` token is allowed to structurally "look" at all the other 196 image patches and absorb their information. 

At the very end of the network, we throw away the 196 image tokens, extract the single updated `[CLS]` token, and feed that single vector into an MLP Classifier to get probabilities.

## 4. Inductive Bias & The Scaling Law Paradigm

### 🎓 DEEP QUESTION ANSWERED
**Q:** *Why do ViTs perform worse than CNNs on small datasets (like ImageNet 1-Million), but obliterate CNNs on massive datasets (JFT-300-Million)?*

**A:** 
A CNN forces the network to look at $3\times3$ grids. It tells the network: *"Trust me, pixels next to each other are related. Use that shortcut."* This is a hardcoded **Inductive Bias**. On small datasets, this saves the CNN—it doesn't have to learn the physics of reality from scratch.

A Vision Transformer tells the network: *"Here are 196 patches. Patch 1 might be related to Patch 2, or maybe Patch 1 is related to Patch 196. I'm not going to force you into a grid. Figure it out yourself via Global Attention."*

On a small dataset, the ViT thrashes around blindly and fails relative to the CNN. But on a massive 300-Million image dataset, the ViT has enough data to organically discover the concept of 2D space on its own. Furthermore, because it is not chained to a strict $3\times3$ grid, it can learn incredibly complex, long-range correlations between the top-left corner and the bottom-right corner that a CNN could never reach without 100 layers.